# E2/E3: Model ladder and continuous competitors (training split)

**Objective.** Fit M0, A_K, M1_K, M2, M3_K, M4_K and LCA_K on the 70% training split, then the exploratory continuous competitors (2D 2PL, 2PL with guessing, 2D with guessing) and their nulls; replication on Forms B and C.

**Data.** Real mode needs the PISA 2025 public-use files processed as described in `data/README.md` (licensed by the OECD and never committed). For Colab, upload `data/processed/math_form*.parquet/.json` and `student_external.parquet` to the same relative paths, or mount Google Drive. Demo mode generates synthetic data. **Demo outputs are not findings.** The *saved results* cells display the PISA 2025 aggregate outputs committed in `results/`.

Errors are raised, never swallowed. Every output is written to `results/` with the form name; outputs from synthetic data are prefixed `demo_`.

In [ ]:
# --- Environment (Colab or local) -------------------------------------------
REPO_URL = ""   # GitHub repository; a private repository needs a token in the URL or an uploaded copy
DEMO = True            # True = synthetic demonstration (no PISA data needed); False = real PISA analysis
import os, sys, subprocess, platform
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("ability-adjusted-mcq-profiling"):
        if not REPO_URL:
            raise SystemExit("Set REPO_URL or upload the repository folder to /content.")
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("ability-adjusted-mcq-profiling")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.exists("configs") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")
sys.path.insert(0, "src"); sys.path.insert(0, "scripts")
import numpy, scipy, pandas, sklearn
print(platform.python_version(), "numpy", numpy.__version__, "scipy", scipy.__version__,
      "pandas", pandas.__version__, "sklearn", sklearn.__version__)
FORM = "demo" if DEMO else "math_formA"
print("Mode:", "SYNTHETIC DEMONSTRATION - not empirical results" if DEMO else "REAL PISA 2025 data")

In [ ]:
import subprocess, sys
def run(stage, form=None):
    cmd = [sys.executable, "scripts/run_experiments.py", "--stage", stage] + (["--demo"] if DEMO else []) \
          + (["--form", form] if form else [])
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-6000:])
    if r.returncode:
        print(r.stderr[-4000:]); raise RuntimeError(f"stage {stage} failed")

In [ ]:
run('fit')

In [ ]:
if not DEMO:
    run('e3b'); run('e3c'); run('replication')   # replication fits Forms B and C
    for f in ['math_formB', 'math_formC']:
        run('competitors', f)

In [ ]:
# Saved PISA 2025 results (aggregate outputs committed to the repository; readable without the licensed data)
import json, pandas as pd
from pathlib import Path
from IPython.display import display, Image, Markdown
def show_json(name, keys=None):
    p = Path("results/metrics") / name
    if not p.exists():
        print("missing", p); return
    d = json.loads(p.read_text(encoding="utf8"))
    print(json.dumps({k: d[k] for k in keys if k in d} if keys else d, indent=1, default=str)[:5000])
def show_csv(name):
    p = Path("results/tables") / name
    display(pd.read_csv(p)) if p.exists() else print("missing", p)
def show_fig(name):
    p = Path("results/figures") / f"{name}.png"
    display(Image(str(p))) if p.exists() else print("missing", p)

In [ ]:
show_csv('math_formA_model_selection_train.csv')

In [ ]:
show_csv('crossform_bic.csv')  # BIC of every model on Forms A, B, C

In [ ]:
show_json('math_formA_E3b_competitors.json', ['bic_train', 'class_contrast_vs_item_properties', 'local_dependence_pairs', 'mdim_null'])
show_json('math_formA_E3c_guess_null.json')

In [ ]:
show_json('math_formA_fit_diagnostics.json')  # class means, SDs, proportions and duration slopes cited in the article

In [ ]:
show_fig('fig05_model_comparison'); show_fig('fig02_04_items_icc')